# 📋 Ejercicio en Clase — Curva ROC, AUC e Índice de Gini
## Diplomado ML en Seguros · Subtema 3.7

---

### Contexto

Eres actuario en **ProtectSalud**, aseguradora de Gastos Médicos Mayores. El área de Suscripción te pide evaluar y comparar tres modelos que predicen si un asegurado tendrá un **evento de alto costo** (hospitalización o cirugía mayor) en los próximos 12 meses.

Los tres modelos ya fueron entrenados por el equipo de datos. Tu tarea es evaluarlos con métricas correctas, construir las curvas ROC, calcular el AUC y el Gini, y recomendar cuál modelo desplegar y con qué umbral.

### Portafolio
- **3,000 pólizas** de GMM individual (generadas en este notebook)
- Tasa de eventos de alto costo: ~9% del portafolio
- **Costo de gestión preventiva (FP):** $4,200 MXN
- **Costo de evento no gestionado (FN):** $210,000 MXN

---

**Instrucciones:** las celdas 🔧 requieren código, las celdas 📝 requieren responder con texto.
No cambies `random_state=2024`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, roc_auc_score, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

np.random.seed(2024)
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})
print('✅ Librerías listas.')

---
## Parte 1 — Cargar el portafolio

Ejecuta esta celda sin modificarla.

In [ ]:
# ── Portafolio ProtectSalud — NO MODIFICAR ───────────────────────────────
np.random.seed(2024)
N = 3000

edad         = np.random.randint(18, 71, N)
genero       = np.random.binomial(1, 0.52, N)
bmi          = np.clip(np.random.normal(27.2, 5.1, N), 16, 45).round(1)
preexistente = np.random.binomial(1, 0.26, N)
fumador      = np.random.binomial(1, 0.15, N)
siniestros_2 = np.random.choice([0,1,2,3], N, p=[0.64,0.23,0.09,0.04])
deducible    = np.random.choice([10,20,50,100], N, p=[0.22,0.33,0.30,0.15])
suma_aseg    = np.random.choice([2,5,10,15,20], N, p=[0.14,0.28,0.33,0.15,0.10])
num_dep      = np.random.choice([0,1,2,3,4], N, p=[0.32,0.27,0.24,0.12,0.05])
zona_riesgo  = np.random.choice([1,2,3,4], N, p=[0.30,0.35,0.22,0.13])

lo = (
    -5.40
    + 0.038 * edad
    + 0.10  * genero
    + 0.09  * np.maximum(bmi - 25, 0)
    + 1.25  * preexistente
    + 0.60  * fumador
    + 0.70  * siniestros_2
    - 0.011 * deducible
    + 0.28  * num_dep
    + 0.22  * zona_riesgo
    + np.random.normal(0, 0.75, N)
)
prob_evento  = 1 / (1 + np.exp(-lo))
evento_alto  = (np.random.rand(N) < prob_evento).astype(int)

df = pd.DataFrame({
    'edad': edad, 'genero': genero, 'bmi': bmi,
    'preexistente': preexistente, 'fumador': fumador,
    'siniestros_2yr': siniestros_2, 'deducible_k': deducible,
    'suma_asegurada_m': suma_aseg, 'num_dependientes': num_dep,
    'zona_riesgo': zona_riesgo, 'evento_alto_costo': evento_alto
})

FEATURES = ['edad','genero','bmi','preexistente','fumador',
            'siniestros_2yr','deducible_k','suma_asegurada_m',
            'num_dependientes','zona_riesgo']
X = df[FEATURES].values
y = df['evento_alto_costo'].values

C_FP = 4_200
C_FN = 210_000

print(f'Portafolio ProtectSalud: {N:,} pólizas')
print(f'Eventos de alto costo:   {evento_alto.sum():,} ({evento_alto.mean()*100:.1f}%)')
print(f'Sin evento:              {(1-evento_alto).sum():,} ({(1-evento_alto).mean()*100:.1f}%)')
print(f'\nCostos: FP=${C_FP:,}  FN=${C_FN:,}  Ratio FN/FP={C_FN/C_FP:.0f}x')
df.head()

---
## Parte 2 — Separar datos y entrenar los modelos

### 🔧 2.1 — Separación train/test

In [ ]:
# 🔧 COMPLETA: separa X e y en train (80%) y test (20%)
# Usa random_state=2024 y stratify=y

# --- TU CÓDIGO AQUÍ ---
X_train, X_test, y_train, y_test = ...

print(f'Train: {len(X_train):,} pólizas | eventos: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Test:  {len(X_test):,} pólizas  | eventos: {y_test.sum():,} ({y_test.mean()*100:.1f}%)')
print('Las tasas deben ser iguales — confirma que stratify funcionó.')

In [ ]:
# Los tres modelos ya están definidos — solo ejecútalos
modelos = {
    'Regresión Logística': Pipeline([('sc', StandardScaler()),
                                      ('m', LogisticRegression(max_iter=1000, random_state=2024))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()),
                                      ('m', RandomForestClassifier(200, max_depth=8,
                                                                   random_state=2024, n_jobs=-1))]),
    'GBM':                 Pipeline([('sc', StandardScaler()),
                                      ('m', GradientBoostingClassifier(
                                           n_estimators=150, max_depth=4,
                                           learning_rate=0.08, random_state=2024))]),
}

probabilidades = {}   # P(evento=1) de cada modelo sobre el test set
predicciones   = {}   # etiquetas con τ=0.5

print('Entrenando...')
for nombre, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    probabilidades[nombre] = pipe.predict_proba(X_test)[:, 1]
    predicciones[nombre]   = pipe.predict(X_test)
    print(f'  {nombre}: entrenado.')
print('✅ Listo.')

---
## Parte 3 — ¿Por qué no basta con la Accuracy?

### 🔧 3.1 — Accuracy del modelo trivial vs los tres modelos

In [ ]:
# 🔧 COMPLETA: calcula la accuracy del modelo trivial
# (el que siempre predice 'sin evento', es decir, siempre 0)
acc_trivial = ...   # --- TU CÓDIGO AQUÍ ---

print(f'Accuracy modelo trivial: {acc_trivial*100:.2f}%  ← siempre predice sin evento')
print(f'Detecta: 0 eventos de alto costo (VP = 0)')
print()

# 🔧 COMPLETA: calcula la accuracy de cada modelo
for nombre, y_pred in predicciones.items():
    acc = ...   # --- TU CÓDIGO AQUÍ ---
    print(f'Accuracy {nombre}: {acc*100:.2f}%')

print()
print('¿La diferencia entre el modelo trivial y los tres modelos parece grande?')
print('¿Un modelo con esa diferencia merece ser desplegado en producción?')

### 📝 Pregunta 3.1

La diferencia de accuracy entre el modelo trivial y el mejor modelo es pequeña. Sin embargo, los modelos detectan VP > 0 y el trivial detecta 0. ¿Por qué la accuracy sola no captura esa diferencia? ¿Qué ocurre cuando la prevalencia es baja (9%)?

> _Escribe tu respuesta aquí_

---
## Parte 4 — Curva ROC y AUC

### 🔧 4.1 — Calcular AUC y Gini para cada modelo

In [ ]:
# 🔧 COMPLETA: para cada modelo calcula el AUC y el Gini
# Recuerda: Gini = 2 × AUC − 1

print(f"{'Modelo':22s}  {'AUC':>8}  {'Gini':>8}  {'Calidad (según guía)'}")
print('-' * 65)

aucs = {}
for nombre, y_prob in probabilidades.items():
    auc  = ...   # --- TU CÓDIGO AQUÍ ---
    gini = ...   # --- TU CÓDIGO AQUÍ ---
    aucs[nombre] = auc

    # Calidad según tabla de la guía
    if auc >= 0.90:   calidad = 'Excelente'
    elif auc >= 0.80: calidad = 'Bueno'
    elif auc >= 0.75: calidad = 'Aceptable'
    elif auc >= 0.65: calidad = 'Pobre'
    else:             calidad = 'Muy pobre'

    print(f"{nombre:22s}  {auc:>8.4f}  {gini:>8.4f}  {calidad}")

### 🔧 4.2 — Graficar las curvas ROC de los tres modelos

In [ ]:
# 🔧 COMPLETA: grafica las tres curvas ROC en una sola figura
# Para cada modelo:
#   1. Calcula fpr, tpr, thresholds con roc_curve()
#   2. Grafica fpr vs tpr con label que incluya nombre, AUC y Gini
# Agrega la diagonal del clasificador aleatorio (línea punteada negra)
# Agrega grid y leyenda

COLORES = {'Regresión Logística': '#0D7490',
           'Random Forest':       '#D97706',
           'GBM':                 '#5B21B6'}

fig, ax = plt.subplots(figsize=(8, 7))

# --- TU CÓDIGO AQUÍ ---


ax.set_xlabel('FPR = FP/(FP+VN)  (Tasa de Falsas Alarmas)')
ax.set_ylabel('TPR = VP/(VP+FN)  (Recall / Sensibilidad)')
ax.set_title('Curvas ROC — ProtectSalud: Predicción de Evento de Alto Costo\n'
             'Cada punto de la curva corresponde a un umbral τ diferente',
             fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 📝 Pregunta 4.2

**a)** ¿Qué modelo tiene la curva ROC más alejada de la diagonal? ¿Coincide con el modelo de mayor AUC?

> _Escribe tu respuesta aquí_

**b)** La curva ROC se calcula sobre el **test set** y no sobre el train set. ¿Por qué es importante esa distinción? ¿Qué pasaría si calcularas el AUC sobre el train set?

> _Escribe tu respuesta aquí_

---
## Parte 5 — Interpretación del AUC y del Gini

### 🔧 5.1 — Interpretación probabilística del AUC

In [ ]:
# 🔧 COMPLETA: selecciona el mejor modelo por AUC
mejor_modelo = max(aucs, key=aucs.get)
auc_mejor    = aucs[mejor_modelo]
gini_mejor   = 2 * auc_mejor - 1

print(f'Mejor modelo: {mejor_modelo}')
print(f'AUC  = {auc_mejor:.4f}')
print(f'Gini = {gini_mejor:.4f}')
print()
print('Interpretación del AUC en términos del negocio:')
print(f'Si tomamos al azar una póliza que SÍ tuvo evento de alto costo')
print(f'y otra que NO lo tuvo, el modelo {mejor_modelo} asigna')
print(f'mayor probabilidad de evento a la correcta el {auc_mejor*100:.1f}% de las veces.')
print()

# Verificación empírica de la interpretación probabilística
# Tomamos 5000 pares aleatorios (una positiva, una negativa) y verificamos
y_prob_mejor = probabilidades[mejor_modelo]

positivos = np.where(y_test == 1)[0]   # índices de eventos reales
negativos = np.where(y_test == 0)[0]   # índices de no-eventos

np.random.seed(42)
n_pares = 5000
idx_pos = np.random.choice(positivos, n_pares)
idx_neg = np.random.choice(negativos, n_pares)

# 🔧 COMPLETA: calcula la fracción de pares donde el modelo asigna
# mayor probabilidad al positivo que al negativo
correctos = ...   # --- TU CÓDIGO AQUÍ ---
# Pista: comparar y_prob_mejor[idx_pos] > y_prob_mejor[idx_neg]

print(f'Verificación empírica con {n_pares:,} pares aleatorios:')
print(f'El modelo discriminó correctamente el {correctos*100:.1f}% de los pares')
print(f'AUC calculado por sklearn:         {auc_mejor*100:.1f}%')
print(f'¿Son similares? {abs(correctos - auc_mejor) < 0.03}')

### 📝 Pregunta 5.1

El AUC calculado por sklearn y la verificación empírica con pares aleatorios deben coincidir aproximadamente. ¿Por qué coinciden? ¿Qué demuestra ese experimento sobre la definición del AUC?

> _Escribe tu respuesta aquí_

---
## Parte 6 — Selección del umbral óptimo

### 🔧 6.1 — Método de Youden

In [ ]:
# 🔧 COMPLETA: para el mejor modelo, calcula el umbral óptimo de Youden
# J = TPR - FPR   →   τ_youden = argmax(J)

fpr_m, tpr_m, thr_m = roc_curve(y_test, probabilidades[mejor_modelo])

J = ...            # --- TU CÓDIGO AQUÍ ---
idx_youden  = ...  # --- TU CÓDIGO AQUÍ ---
tau_youden  = ...  # --- TU CÓDIGO AQUÍ ---

print(f'Umbral Youden:')
print(f'  τ* = {tau_youden:.4f}')
print(f'  TPR en τ*: {tpr_m[idx_youden]*100:.1f}%')
print(f'  FPR en τ*: {fpr_m[idx_youden]*100:.1f}%')
print(f'  J = TPR - FPR = {J[idx_youden]:.4f}')

In [ ]:
# 🔧 COMPLETA: calcula el umbral óptimo basado en costos
# τ* = C_FP / (C_FP + C_FN)

tau_costos = ...   # --- TU CÓDIGO AQUÍ ---

print(f'Umbral basado en costos:')
print(f'  C_FP = ${C_FP:,} MXN  (gestión preventiva innecesaria)')
print(f'  C_FN = ${C_FN:,} MXN  (evento de alto costo no gestionado)')
print(f'  τ* = {C_FP:,} / ({C_FP:,} + {C_FN:,}) = {tau_costos:.4f}')
print()
print(f'Comparación de umbrales:')
print(f'  τ = 0.50 (default sklearn): {(probabilidades[mejor_modelo] >= 0.50).sum()} alarmas')
print(f'  τ = {tau_youden:.3f} (Youden):          {(probabilidades[mejor_modelo] >= tau_youden).sum()} alarmas')
print(f'  τ = {tau_costos:.4f} (costos):         {(probabilidades[mejor_modelo] >= tau_costos).sum()} alarmas')

In [ ]:
# 🔧 COMPLETA: evalúa el mejor modelo con los tres umbrales
# Para cada umbral calcula: VP, FP, FN, Recall, Precisión, y Valor Neto

print(f"{'Umbral':>10}  {'Alarmas':>8}  {'VP':>5}  {'FP':>5}  {'FN':>5}  "
      f"{'Recall':>8}  {'Precisión':>10}  {'Valor neto':>12}")
print('-' * 80)

PROB_EXITO = 0.35   # probabilidad de retener el evento con gestión preventiva

for tau, nombre_tau in [(0.50, 'default'),
                         (tau_youden, 'Youden'),
                         (tau_costos, 'Costos')]:

    y_pred_tau = (probabilidades[mejor_modelo] >= tau).astype(int)
    cm = confusion_matrix(y_test, y_pred_tau)
    vn_t, fp_t, fn_t, vp_t = cm.ravel()

    rec_t  = vp_t / (vp_t + fn_t) if (vp_t + fn_t) > 0 else 0
    prec_t = vp_t / (vp_t + fp_t) if (vp_t + fp_t) > 0 else 0

    # Valor neto = ahorro por eventos evitados - costo de gestiones
    # --- TU CÓDIGO AQUÍ ---
    valor_neto = ...

    alarmas = y_pred_tau.sum()
    print(f"  {tau:.4f} ({nombre_tau:>7})  {alarmas:>8}  {vp_t:>5}  "
          f"{fp_t:>5}  {fn_t:>5}  {rec_t*100:>7.1f}%  "
          f"{prec_t*100:>9.1f}%  ${valor_neto:>10,.0f}")

### 📝 Pregunta 6

**a)** El umbral basado en costos es mucho más bajo que τ=0.5. ¿Por qué? Explica usando los valores de C_FP y C_FN de este portafolio.

> _Escribe tu respuesta aquí_

**b)** Con el umbral de Youden, ¿qué porcentaje de los eventos reales de alto costo logra detectar el modelo? ¿Es ese Recall suficiente para el área de Suscripción o deberían bajar más el umbral?

> _Escribe tu respuesta aquí_

---
## Parte 7 — Visualización final y recomendación

### 🔧 7.1 — Gráfica del J de Youden

In [ ]:
# 🔧 COMPLETA: genera las dos gráficas siguientes en una figura con 2 paneles

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Selección de Umbral — {mejor_modelo}', fontweight='bold')

# Panel 1: TPR, FPR y J de Youden en función de τ
ax = axes[0]
# 🔧 COMPLETA:
#   - Grafica thr_m vs tpr_m[:-1] (color teal, label 'TPR (Recall)')
#   - Grafica thr_m vs fpr_m[:-1] (color red, label 'FPR (Falsas alarmas)')
#   - Grafica thr_m vs J[:-1] (color purple, linestyle '--', label 'J = TPR - FPR')
#   - Marca con axvline el tau_youden (color orange) y tau_costos (color green)
#   - Agrega labels, título y leyenda
# --- TU CÓDIGO AQUÍ ---

ax.set_xlabel('Umbral τ')
ax.set_title('Métricas por Umbral\nBuscar máximo de J', fontweight='bold')

# Panel 2: distribución de probabilidades por clase
ax = axes[1]
# 🔧 COMPLETA:
#   - Histograma de probabilidades para y_test==0 (color verde, alpha=0.65)
#   - Histograma de probabilidades para y_test==1 (color rojo, alpha=0.65)
#   - Línea vertical en tau_youden (color naranja)
#   - Línea vertical en 0.5 (color negro punteado)
#   - Usa density=True para que las áreas sean comparables
# --- TU CÓDIGO AQUÍ ---

ax.set_xlabel('P(evento alto costo)')
ax.set_title('Distribución de Probabilidades\npor Clase Real', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Parte 8 — Preguntas finales de discusión

### 📝 8.1 — Recomendación al área de Suscripción

**a) ¿Cuál modelo recomiendas desplegar?** Justifica usando AUC y Gini. ¿Por qué no usas solo la accuracy para tomar esta decisión?

> _Escribe tu respuesta aquí_

---

**b) ¿Qué umbral τ recomiendas usar en producción?** El área de Gestión de Salud puede atender un máximo de 60 casos por semana. Con 3,000 pólizas que se evalúan trimestralmente (750 por semana), ¿qué umbral te da aproximadamente esa cantidad de alarmas?

> _Escribe tu respuesta aquí_

In [ ]:
# 🔧 8.2 BONUS — Encuentra el τ que produce ~60 alarmas por semana
# El test set tiene 600 pólizas (20% de 3,000)
# Escala: 60/750 * 600 = 48 alarmas en el test set

objetivo_alarmas_test = int(60 / 750 * len(X_test))
print(f'Objetivo de alarmas en test set: {objetivo_alarmas_test}')

# 🔧 COMPLETA: encuentra el τ que produce ese número de alarmas
# Pista: ordena y_prob de mayor a menor y toma el percentil correcto
# --- TU CÓDIGO AQUÍ ---


---
## Resumen del ejercicio

| Lo que practicaste | Dónde |
|--------------------|-------|
| Accuracy engañosa con prevalencia baja | Parte 3 |
| Calcular AUC y Gini con sklearn | Parte 4 |
| Graficar curvas ROC de tres modelos | Parte 4 |
| Interpretación probabilística del AUC | Parte 5 |
| Verificación empírica del AUC con pares | Parte 5 |
| Umbral Youden: τ = argmax(TPR − FPR) | Parte 6 |
| Umbral por costos: τ* = C_FP/(C_FP+C_FN) | Parte 6 |
| Análisis económico por umbral | Parte 6 |
| Selección de modelo y τ para producción | Parte 8 |